In [6]:
%pip install python-dotenv
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 38.6 MB/s  0:00:00
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)

   ------------- -------------------------- 1/3 [greenlet]
   ------------- -------------------------- 1/3 [greenlet]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/

In [1]:
import os
import sys
from datetime import date, datetime
from dotenv import load_dotenv
# Configuración de rutas
sys.path.append("../src")
# from repositories import *
from models import *



load_dotenv()

from domain.config import DB_URL
from repositories import UnitOfWorkFactory
from models import (
    Region, Player, Team, VideoGameType, 
    TournamentMode, TournamentStatus, Tournament, Match
)
from repositories import (
    PlayerRepository, TeamRepository, TournamentRepository, 
    RegionRepository, VideoGameTypeRepository
)

print(f"Entorno listo. Conectado a: {DB_URL}")

# Inicialización de la fábrica
uow_factory = UnitOfWorkFactory(DB_URL)

# La función de ayuda ahora llama a .create()
def get_uow():
    return uow_factory.create()

Entorno listo. Conectado a: sqlite:///:memory:


In [ ]:
# --- FORZAR CREACIÓN DE TABLAS (Añadir al final de la Celda 1) ---
from domain.db import Base

# Usamos el engine de la factoría para crear lo que falte
Base.metadata.create_all(bind=uow_factory.engine)

print("Tablas verificadas/creadas en la base de datos.")

Tablas verificadas/creadas en la base de datos.


PlayerTeam + TeamTournament


In [ ]:
print("--- TESTING Intermediate Repositories (Many-to-Many) ---")

with get_uow() as uow:
    from repositories import PlayerTeamRepository, TeamTournamentRepository
    from models import PlayerTeam, TeamTournament
    from datetime import date
    
    pt_repo = PlayerTeamRepository(uow, PlayerTeam)
    tt_repo = TeamTournamentRepository(uow, TeamTournament)
    
    # --- PASO 0: LIMPIEZA (Para evitar el UNIQUE constraint error) ---
    # Borramos registros previos de estas tablas para que el test empiece de cero
    uow.session.query(PlayerTeam).delete()
    uow.session.query(TeamTournament).delete()
    uow.commit() 
    print("🧹 Tablas intermedias limpiadas para el test.")

    # --- 1. TEST PLAYER-TEAM ---
    # CREACIÓN (Ahora ya no fallará porque acabamos de borrar el anterior)
    vinc_pt = PlayerTeam(id_player=1, id_team=1, registered_at=date.today())
    pt_repo.add(vinc_pt)
    uow.commit()
    print("[OK] PlayerTeam: Registro creado.")

    # LECTURA ALL
    todos_pt = pt_repo.get_all()
    print(f"[OK] PlayerTeam: Leídos {len(todos_pt)} registros.")

    # --- 2. TEST TEAM-TOURNAMENT ---
    vinc_tt = TeamTournament(id_team=1, id_tournament=1, points_obtained=500)
    tt_repo.add(vinc_tt)
    uow.commit()
    print("[OK] TeamTournament: Registro creado.")

    # BORRADO (Opcional: puedes borrarlo al final para dejar la DB limpia)
    # pt_repo.delete(vinc_pt)
    # uow.commit()
    # print("[OK] Limpieza final realizada.")

print("\n¡Test completado con éxito y sin errores de duplicados!")

--- TESTING Intermediate Repositories (Many-to-Many) ---
🧹 Tablas intermedias limpiadas para el test.
[OK] PlayerTeam: Registro creado.
[OK] PlayerTeam: Leídos 1 registros.
[OK] TeamTournament: Registro creado.

¡Test completado con éxito y sin errores de duplicados!


PlayerProfile + Match

In [ ]:
print("--- TESTING Profile & Match ---")

with get_uow() as uow:
    from repositories import PlayerProfileRepository, MatchRepository, TeamRepository
    from models import PlayerProfile, Match, Team
    from datetime import datetime, date
    
    # 1. Limpieza de seguridad
    uow.session.query(Match).delete()
    uow.session.query(PlayerProfile).delete()
    
    # 2. Aseguramos que existan los equipos con los campos que pide tu DB
    team_repo = TeamRepository(uow, Team)
    for tid, tname in [(1, "Team Alpha"), (2, "Team Beta")]:
        if not team_repo.get(tid):
            team_repo.add(Team(id_team=tid, name=tname, creation_date=date.today(), id_region=1))
    uow.commit()

    # 3. Instanciamos repositorios
    profile_repo = PlayerProfileRepository(uow, PlayerProfile)
    match_repo = MatchRepository(uow, Match)
    
    # 4. Player Profile (Basado en tus estadísticas)
    perfil = PlayerProfile(
        id_player=1, 
        ranking_position=1,
        total_matches=0,
        total_wins=0
    )
    profile_repo.add(perfil)
    
    # 5. Match (CORREGIDO con tus nombres de columna)
    partido = Match(
        id_tournament=1,
        id_team_one=1,        # Antes era id_team_home
        id_team_two=2,        # Antes era id_team_away
        score_team_one=10,    # Antes era score_home
        score_team_two=5,     # Antes era score_away
        match_date=datetime.now(),
        round_number=1,       # ¡Obligatorio en tu modelo!
        duration_minutes=45   # ¡Obligatorio en tu modelo!
    )
    match_repo.add(partido)
    
    uow.commit()

    # 6. Verificación
    partidos_torneo = match_repo.get_matches_by_tournament(tournament_id=1)
    
    print(f"Perfil de estadísticas creado para Player 1.")
    print(f"Match registrado: {partido.id_team_one} vs {partido.id_team_two}")
    print(f"Total partidos en el torneo 1: {len(partidos_torneo)}")

--- TESTING Profile & Match ---
Perfil de estadísticas creado para Player 1.
Match registrado: 1 vs 2
Total partidos en el torneo 1: 1


TournamentMode + TournamentStatus

In [ ]:
print("--- TESTING Catalog Repositories ---")
with get_uow() as uow:
    from repositories import TournamentModeRepository, TournamentStatusRepository
    from models import TournamentMode, TournamentStatus
    
    # PASO CORRECTO: Pasar 'uow' en lugar de 'uow.session'
    mode_repo = TournamentModeRepository(uow, TournamentMode)
    status_repo = TournamentStatusRepository(uow, TournamentStatus)
    
    # Nota: Asegúrate de que el método se llame 'get_all()' o 'list()' 
    # según lo que definiste en tu clase base SqlAlchemyRepository.
    modos = mode_repo.get_all()
    estados = status_repo.get_all()
    
    print(f"[OK] Catalogs: {len(modos)} modos y {len(estados)} estados leídos.")

--- TESTING Catalog Repositories ---
[OK] Catalogs: 1 modos y 1 estados leídos.


Region

In [ ]:
print("--- TESTING RegionRepository ---")
with get_uow() as uow:
    # Cambio crítico: pasar 'uow' directo
    repo = RegionRepository(uow, Region)
    
    # 1. Creation
    nueva_reg = Region(name="Latam North", country_code="LN")
    repo.add(nueva_reg)
    uow.commit()
    reg_id = nueva_reg.id_region
    print(f"[OK] Creation: ID {reg_id}")

    # 2. Reading by ID
    reg = repo.get(reg_id)
    print(f"[OK] Read by ID: {reg.name}")

    # 3. Reading all
    # Si 'get_all' falla, asegúrate de haberlo definido en abstract_repository
    todas = repo.get_all()
    print(f"[OK] Read all: {len(todas)} regiones encontradas")

    # 4. Updating
    reg.name = "Latam North Updated"
    uow.commit()
    print(f"[OK] Updating: {repo.get(reg_id).name}")

    # 5. Specific Query (count_players)
    # Este método debe estar implementado en region_repository.py
    count = repo.count_players(reg_id)
    print(f"[OK] Specific Query (count_players): {count} jugadores")

--- TESTING RegionRepository ---
[OK] Creation: ID 3
[OK] Read by ID: Latam North
[OK] Read all: 3 regiones encontradas
[OK] Updating: Latam North Updated
[OK] Specific Query (count_players): 0 jugadores


Player

In [ ]:
print("--- TESTING PlayerRepository ---")
with get_uow() as uow:
    from repositories import PlayerRepository, TeamRepository
    from models import Player, Team, PlayerTeam
    from datetime import datetime
    
    # 0. Preparación y Limpieza
    # Borramos registros previos de la tabla intermedia y jugadores para este test
    uow.session.query(PlayerTeam).delete()
    uow.session.query(Player).delete()
    
    # Aseguramos que el Equipo 1 exista para la operación de dominio
    team_repo = TeamRepository(uow, Team)
    if not team_repo.get(1):
        team_repo.add(Team(id_team=1, name="Team Alpha", creation_date=datetime.now(), id_region=1))
    
    uow.commit()

    # 1. Instanciación del repositorio
    player_repo = PlayerRepository(uow, Player)
    
    # 2. Creación de jugadores
    # Usamos datetime para coincidir con la definición de tu modelo player.py
    p1 = Player(
        id_player=1,
        nickname="Gamer1", 
        email="g1@test.com", 
        birth_date=datetime(1990, 1, 1), 
        registration_date=datetime.now(), 
        id_region=1
    )
    p2 = Player(
        id_player=2,
        nickname="Gamer2", 
        email="g2@test.com", 
        birth_date=datetime(1992, 2, 2), 
        registration_date=datetime.now(), 
        id_region=1
    )
    player_repo.add(p1)
    player_repo.add(p2)
    uow.commit()
    print("[OK] Creation: Jugadores creados")

    # 3. Paginación
    page = player_repo.get_paginated(page=1, page_size=1)
    print(f"[OK] Pagination: {len(page)} registro(s) en pág 1")

    # 4. Operación de Dominio (Vinculación Directa)
    # Ya no fallará por UNIQUE constraint porque limpiamos al inicio
    player_repo.add_player_to_team_direct(player_id=p1.id_player, team_id=1)
    uow.commit()
    print("[OK] Domain Op: Jugador vinculado a equipo")

    # 5. Borrado
    player_repo.delete(p2)
    uow.commit()
    print("[OK] Deleting: Gamer2 eliminado")

# --- VERIFICACIÓN EXTRA (Opcional) ---
with get_uow() as uow:
    from repositories import RegionRepository
    from models import Region
    reg_repo = RegionRepository(uow, Region)
    count = reg_repo.count_players(1)
    print(f"--- VERIFICACIÓN POST-TEST ---")
    print(f"[OK] Conteo actualizado en Región 1: {count} jugadores")

--- TESTING PlayerRepository ---
[OK] Creation: Jugadores creados
[OK] Pagination: 1 registro(s) en pág 1
[OK] Domain Op: Jugador vinculado a equipo
[OK] Deleting: Gamer2 eliminado
--- VERIFICACIÓN POST-TEST ---
[OK] Conteo actualizado en Región 1: 1 jugadores


Tournament + VideoGameType

In [ ]:
print("--- TESTING Tournament & VideoGameType ---")

with get_uow() as uow:
    from repositories import VideoGameTypeRepository, TournamentRepository
    from models import VideoGameType, Tournament
    from datetime import date
    
    vgt_repo = VideoGameTypeRepository(uow, VideoGameType)
    tour_repo = TournamentRepository(uow, Tournament)
    
    # 1. Crear tipo de juego
    vgt = VideoGameType(name="FPS", description="First Person Shooter")
    vgt_repo.add(vgt)
    uow.commit()
    
    # 2. Crear Torneo con end_date incluido
    nuevo_torneo = Tournament(
        name="Blast Premier 2026",
        start_date=date(2026, 6, 1),
        end_date=date(2026, 6, 15), # Se añade este campo para cumplir con la restricción
        id_video_game_type=vgt.id_video_game_type,
        id_region=1,
        id_tournament_mode=1,
        id_tournament_status=1,
        prize_pool=50000.00,
        max_teams=16
    )
    tour_repo.add(nuevo_torneo)
    uow.commit()
    
    # 3. Consulta específica
    try:
        top = tour_repo.get_top_players(nuevo_torneo.id_tournament, limit=3)
        print(f"[OK] Specific Query (Top Players): Ejecutada correctamente")
    except AttributeError:
        print("[ERROR] El método 'get_top_players' no existe en TournamentRepository")

--- TESTING Tournament & VideoGameType ---
[OK] Specific Query (Top Players): Ejecutada correctamente


In [ ]:
print("--- VERIFYING ALEMBIC MIGRATION (last_update) ---")
with get_uow() as uow:
    reg = uow.session.query(Region).first()
    if hasattr(reg, 'last_update'):
        print(f"✅ Campo 'last_update' detectado: {reg.last_update}")
    else:
        print("❌ Error: No se detecta el campo last_update")

--- VERIFYING ALEMBIC MIGRATION (last_update) ---
✅ Campo 'last_update' detectado: 2026-05-11 02:30:51.862877
